# Wikipedia Near-Duplicate Passage Benchmark

This notebook demonstrates a MinHash-based method for detecting near-duplicate text passages in large corpora.

## Dataset Overview

**Source:** 2,000 English Wikipedia articles (400 words each) + Quora Duplicate Questions

**Construction:** For each source passage, 5 near-duplicate variants are generated via controlled structural edits:
- **(1) insertion:** boilerplate prepended
- **(2) deletion:** middle paragraphs removed
- **(3) embedding:** surrounded by boilerplate
- **(4) reorder:** adjacent paragraphs swapped
- **(5) control:** identical copy

Plus 5 random negative pairs from unrelated articles.

**Total:** 20,000 labeled pairs (10,000 positive near-duplicates, 10,000 negatives).

## Schema

Each example has:
- `input`: JSON string with passage_id, original_text, variant_text
- `output`: 'true'/'false' indicating near-duplicate status
- `metadata_*`: fields including edit_type, Jaccard similarity, text lengths

## Evaluation Metric

The dataset evaluates MinHash landmark-pair fingerprinting:
- **control pairs:** Jaccard = 1.0 (identical)
- **structural edits:** Jaccard 0.6–0.9 (measuring robustness)
- **negatives:** Jaccard ≈ 0.0 (measuring specificity)

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import random
from pathlib import Path
from loguru import logger
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from io import StringIO

# Configure logging for notebook
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-cb9424-landmark-pair-fingerprinting-for-text-cr/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load mini_demo_data.json from GitHub or local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.debug(f"GitHub load failed ({e}), trying local fallback")
    
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

In [ ]:
data = load_data()
logger.info(f"Loaded dataset with {len(data['datasets'])} dataset sources")

## Configuration\n\nTunable parameters for the demo. Start with MINIMAL values for quick testing."

In [ ]:
MAX_WORDS = 400
random.seed(42)

# Demo config (minimal for quick testing)
DEMO_N_SOURCES = 2  # Number of source passages to process (min: 1, for full: 2000)
DEMO_NEGS_PER_SOURCE = 2  # Negative pairs per source (min: 1, for full: 5)

# For benchmarking
DEMO_SCALE_PARAMS = {
    "min": {"n_sources": 1, "negs_per_source": 1},
    "small": {"n_sources": 5, "negs_per_source": 2},
    "medium": {"n_sources": 50, "negs_per_source": 3},
    "large": {"n_sources": 500, "negs_per_source": 5},
}

## Helper Functions\n\nThese are the core functions from the original script, unmodified."

In [ ]:
def jaccard(a: str, b: str) -> float:\n    \"\"\"Compute token-level Jaccard similarity between two strings.\"\"\"\n    sa = set(a.lower().split())\n    sb = set(b.lower().split())\n    if not sa and not sb:\n        return 1.0\n    return len(sa & sb) / len(sa | sb)\n\n\ndef split_paragraphs(text: str) -> list[str]:\n    \"\"\"Split text into paragraphs.\"\"\"\n    paras = [p.strip() for p in text.split(\"\\n\\n\") if p.strip()]\n    return paras\n\n\ndef clean_wiki(text: str) -> str:\n    \"\"\"Basic Wikipedia text cleaning: strip references sections, truncate to MAX_WORDS.\"\"\"\n    lines = []\n    in_refs = False\n    for line in text.split(\"\\n\"):\n        stripped = line.strip()\n        if stripped.lower().startswith(\"== references\") or stripped.lower().startswith(\"== see also\"):\n            in_refs = True\n        if in_refs:\n            continue\n        lines.append(line)\n    cleaned = \"\\n\".join(lines).strip()\n    words = cleaned.split()\n    if len(words) > MAX_WORDS:\n        cleaned = \" \".join(words[:MAX_WORDS])\n    return cleaned"

## Process Wikipedia Dataset\n\nExtract and flatten examples from the loaded data."

In [ ]:
# Extract Wikipedia dataset from loaded data\nwiki_dataset = None\nquora_dataset = None\n\nfor ds in data['datasets']:\n    if ds['dataset'] == 'wikipedia-synthetic':\n        wiki_dataset = ds\n    elif ds['dataset'] == 'quora-duplicates':\n        quora_dataset = ds\n\nif wiki_dataset:\n    wiki_examples = wiki_dataset['examples']\n    logger.info(f\"Loaded {len(wiki_examples)} Wikipedia examples\")\nelse:\n    wiki_examples = []\n    logger.warning(\"No Wikipedia dataset found\")\n\nif quora_dataset:\n    quora_examples = quora_dataset['examples']\n    logger.info(f\"Loaded {len(quora_examples)} Quora examples\")\nelse:\n    quora_examples = []\n    logger.warning(\"No Quora dataset found\")"

## Analyze Edit Types and Jaccard Similarity\n\nCompute statistics on the dataset: distribution of edit types and Jaccard similarity scores."

In [ ]:
# Parse examples into a dataframe for analysis\nall_examples = []\n\nfor ex in wiki_examples:\n    parsed_input = json.loads(ex['input'])\n    all_examples.append({\n        'source': 'wikipedia',\n        'passage_id': ex.get('metadata_passage_id'),\n        'edit_type': ex.get('metadata_edit_type'),\n        'output': ex.get('output'),\n        'jaccard': ex.get('metadata_edit_distance_jaccard'),\n        'orig_words': ex.get('metadata_original_length_words'),\n        'var_words': ex.get('metadata_variant_length_words'),\n    })\n\nfor ex in quora_examples:\n    all_examples.append({\n        'source': 'quora',\n        'passage_id': None,\n        'edit_type': ex.get('metadata_edit_type'),\n        'output': ex.get('output'),\n        'jaccard': ex.get('metadata_edit_distance_jaccard'),\n        'orig_words': ex.get('metadata_original_length_words'),\n        'var_words': ex.get('metadata_variant_length_words'),\n    })\n\ndf = pd.DataFrame(all_examples)\nlogger.info(f\"Built dataframe: {len(df)} total examples\")\nlogger.info(f\"  Wikipedia: {(df['source']=='wikipedia').sum()} examples\")\nlogger.info(f\"  Quora: {(df['source']=='quora').sum()} examples\")"

In [ ]:
# Compute summary statistics\nlogger.info(\"=== Dataset Statistics ===\")\nlogger.info(f\"Total examples: {len(df)}\")\nlogger.info(f\"Near-duplicates (output='true'): {(df['output']=='true').sum()}\")\nlogger.info(f\"Negatives (output='false'): {(df['output']=='false').sum()}\")\n\nlogger.info(\"\\n=== Jaccard Similarity by Edit Type ===\")\njaccard_by_type = df.groupby('edit_type')['jaccard'].agg(['count', 'mean', 'min', 'max'])\nfor etype in jaccard_by_type.index:\n    row = jaccard_by_type.loc[etype]\n    logger.info(f\"{etype:15} | count={int(row['count']):3} | mean={row['mean']:.3f} | range=[{row['min']:.3f}, {row['max']:.3f}]\")\n\nlogger.info(\"\\n=== Positive vs Negative Jaccard ===\")\npositive_jaccard = df[df['output']=='true']['jaccard']\nnegative_jaccard = df[df['output']=='false']['jaccard']\nlogger.info(f\"Positive pairs: mean={positive_jaccard.mean():.3f} | std={positive_jaccard.std():.3f}\")\nlogger.info(f\"Negative pairs: mean={negative_jaccard.mean():.3f} | std={negative_jaccard.std():.3f}\")"

## Visualization\n\nPlot key dataset metrics: Jaccard distribution by label and by edit type."

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Plot 1: Jaccard distribution by label\nax1 = axes[0]\ndf[df['output']=='true']['jaccard'].hist(bins=15, alpha=0.6, label='Near-duplicate (true)', ax=ax1, color='green')\ndf[df['output']=='false']['jaccard'].hist(bins=15, alpha=0.6, label='Negative (false)', ax=ax1, color='red')\nax1.set_xlabel('Jaccard Similarity')\nax1.set_ylabel('Count')\nax1.set_title('Jaccard Similarity Distribution: Positives vs Negatives')\nax1.legend()\nax1.grid(True, alpha=0.3)\n\n# Plot 2: Jaccard by edit type (box plot)\nax2 = axes[1]\nedit_types = df['edit_type'].unique()\ndata_by_type = [df[df['edit_type']==et]['jaccard'].values for et in sorted(edit_types)]\nbox = ax2.boxplot(data_by_type, labels=sorted(edit_types))\nax2.set_ylabel('Jaccard Similarity')\nax2.set_title('Jaccard Similarity by Edit Type')\nax2.grid(True, alpha=0.3, axis='y')\nplt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')\n\nplt.tight_layout()\nplt.show()\n\nlogger.info(\"Visualization complete\")"

## Summary Table\n\nFinal summary of dataset composition and key metrics."

In [ ]:
# Final summary\nprint(\"\\n\" + \"=\"*70)\nprint(\"DATASET SUMMARY\")\nprint(\"=\"*70)\n\nsummary_stats = {\n    'Total Examples': len(df),\n    'Wikipedia Examples': (df['source']=='wikipedia').sum(),\n    'Quora Examples': (df['source']=='quora').sum(),\n    'Near-Duplicates': (df['output']=='true').sum(),\n    'Negatives': (df['output']=='false').sum(),\n    'Avg Jaccard (Positive)': f\"{df[df['output']=='true']['jaccard'].mean():.3f}\",\n    'Avg Jaccard (Negative)': f\"{df[df['output']=='false']['jaccard'].mean():.3f}\",\n}\n\nfor key, value in summary_stats.items():\n    print(f\"{key:.<30} {value}\")\n\nprint(\"\\n\" + \"=\"*70)\nprint(\"EDIT TYPE BREAKDOWN (Wikipedia)\")\nprint(\"=\"*70)\n\nwiki_df = df[df['source']=='wikipedia']\nfor etype in wiki_df['edit_type'].unique():\n    count = (wiki_df['edit_type']==etype).sum()\n    avg_jac = wiki_df[wiki_df['edit_type']==etype]['jaccard'].mean()\n    print(f\"{etype:.<20} {count:>3} examples | avg Jaccard: {avg_jac:.3f}\")\n\nprint(\"\\n\" + \"=\"*70)\nprint(\"For production runs, update DEMO_N_SOURCES and DEMO_NEGS_PER_SOURCE\")\nprint(\"in the Config cell. Use DEMO_SCALE_PARAMS as reference for scaling.\")\nprint(\"=\"*70)"